# Transit Connectivity Analysis — Dallas County

**Question:** What percentage of workers can actually reach their job by public transit?

**Method:** Match LODES commute data (where people work) with GTFS route data (where buses go). For each worker, check if a transit path exists from home to work — 0.5mi walk, up to 2 transfers.

**Data:** EPA SLD (demographics), LODES 2021 (commute flows), DART GTFS 2021 (bus routes)

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from collections import defaultdict
import pygris
from math import atan2, degrees

WALK_RADIUS_DEG = 0.5 / 69
TRANSFER_RADIUS_DEG = 0.1 / 69

## 1. Load Data

In [ ]:
sld = pd.read_csv('EPA_SmartLocationDatabase_V3_Jan_2021_Final.csv')
sld['GEOID'] = (sld['STATEFP'].astype(str).str.zfill(2) +
                sld['COUNTYFP'].astype(str).str.zfill(3) +
                sld['TRACTCE'].astype(str).str.zfill(6) +
                sld['BLKGRPCE'].astype(str).str.zfill(1))
sld = sld[(sld['TotPop'] > 50) & (sld['D5BR'] != -99999)].copy()
sld['D4A'] = sld['D4A'].replace(-99999, np.nan)
for col in ['D4C', 'D4D', 'D4E']:
    sld[col] = sld[col].replace(-99999, 0)
sld['transit_ratio'] = np.where(sld['D5AR'] > 0, sld['D5BR'] / sld['D5AR'], 0)

od = pd.read_csv('tx_od_main_JT00_2021.csv.gz', dtype={'w_geocode': str, 'h_geocode': str})
od['home_bg'] = od['h_geocode'].str[:12]
od['work_bg'] = od['w_geocode'].str[:12]
lodes = od.groupby(['home_bg', 'work_bg']).agg(
    total_workers=('S000', 'sum'), low_wage=('SE01', 'sum')
).reset_index()
lodes = lodes[lodes['home_bg'] != lodes['work_bg']]

dallas_od = lodes[lodes['home_bg'].str[:5] == '48113'].copy()
print(f'Dallas commuters: {dallas_od["total_workers"].sum():,}')

## 2. Build Transit Connectivity
Map each block group to walkable bus stops, then determine which block groups can reach each other via direct route or transfers.

In [ ]:
gtfs = 'V248-160-161-20210517'
stops = pd.read_csv(f'{gtfs}/stops.txt')
trips = pd.read_csv(f'{gtfs}/trips.txt')
stop_times = pd.read_csv(f'{gtfs}/stop_times.txt')
routes = pd.read_csv(f'{gtfs}/routes.txt')

trip_routes = trips[['trip_id', 'route_id']].drop_duplicates()
sr = (stop_times[['trip_id', 'stop_id']]
      .merge(trip_routes, on='trip_id')[['stop_id', 'route_id']]
      .drop_duplicates())
sr = sr.merge(stops[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id')
sr = sr.merge(routes[['route_id', 'route_short_name']], on='route_id')

stop_route_map = sr.groupby('stop_id')['route_short_name'].apply(set).to_dict()
stop_locs = sr[['stop_id', 'stop_lat', 'stop_lon']].drop_duplicates('stop_id')

geo = pygris.block_groups(state='48', county='113', year=2020)
geo['GEOID'] = geo['GEOID'].astype(str).str.zfill(12)
geo = geo.to_crs('EPSG:4326')
geo['lat'] = geo.geometry.centroid.y
geo['lon'] = geo.geometry.centroid.x

print(f'DART routes: {sr["route_short_name"].nunique()}, Stops: {len(stop_locs)}, Block groups: {len(geo)}')

In [ ]:
# Map BGs to routes, find transfers, build reachability
stop_tree = cKDTree(stop_locs[['stop_lat', 'stop_lon']].values)
stop_ids = stop_locs['stop_id'].values

bg_routes = {}
for _, row in geo.iterrows():
    nearby = stop_tree.query_ball_point([row['lat'], row['lon']], WALK_RADIUS_DEG)
    rset = set()
    for idx in nearby:
        rset.update(stop_route_map.get(stop_ids[idx], set()))
    if rset:
        bg_routes[row['GEOID']] = rset

route_bgs = defaultdict(set)
for bg_id, rset in bg_routes.items():
    for r in rset:
        route_bgs[r].add(bg_id)

route_stop_coords = defaultdict(list)
for _, row in sr.iterrows():
    route_stop_coords[row['route_short_name']].append((row['stop_lat'], row['stop_lon']))

transfers = defaultdict(set)
route_names = list(route_stop_coords.keys())
for i, r1 in enumerate(route_names):
    tree1 = cKDTree(np.array(route_stop_coords[r1]))
    for j, r2 in enumerate(route_names):
        if i >= j: continue
        for pt in route_stop_coords[r2]:
            d, _ = tree1.query(pt, k=1)
            if d < TRANSFER_RADIUS_DEG:
                transfers[r1].add(r2)
                transfers[r2].add(r1)
                break

bg_reach = {}
for bg_id, rset in bg_routes.items():
    routes_1t = set(rset)
    for r in rset:
        routes_1t.update(transfers.get(r, set()))
    routes_2t = set(routes_1t)
    for r in routes_1t:
        routes_2t.update(transfers.get(r, set()))
    reachable = set()
    for r in routes_2t:
        reachable.update(route_bgs.get(r, set()))
    reachable.discard(bg_id)
    bg_reach[bg_id] = reachable

print(f'BGs with transit: {len(bg_routes)} / {len(geo)}')
print(f'Transfer pairs: {sum(len(v) for v in transfers.values()) // 2}')

## 3. Results

In [ ]:
dallas_od['connected'] = [
    w in bg_reach.get(h, set())
    for h, w in zip(dallas_od['home_bg'], dallas_od['work_bg'])
]

total = dallas_od['total_workers'].sum()
connected = dallas_od[dallas_od['connected']]['total_workers'].sum()
stranded = dallas_od[~dallas_od['connected']]['total_workers'].sum()

print(f'Total internal commuters:     {total:>10,}')
print(f'CAN reach work by transit:    {connected:>10,}  ({connected/total:.0%})')
print(f'CANNOT reach work by transit: {stranded:>10,}  ({stranded/total:.0%})')

## 4. Proposed Route Corridors
Group stranded workers by job center and compass direction. Each direction = a proposed linear route.

In [ ]:
stranded_df = dallas_od[~dallas_od['connected']].copy()
stranded_df['home_tract'] = stranded_df['home_bg'].str[:11]
stranded_df['work_tract'] = stranded_df['work_bg'].str[:11]

top_jc = (stranded_df.groupby('work_tract')['total_workers'].sum()
          .nlargest(5).reset_index())
top_jc.columns = ['Job Center Tract', 'Stranded Workers']
print('Top 5 job centers stranded workers need to reach:')
print(top_jc.to_string(index=False))

In [ ]:
geo['tract'] = geo['GEOID'].str[:11]
tract_cents = geo.groupby('tract').agg(lat=('lat', 'mean'), lon=('lon', 'mean')).to_dict('index')

def compass_dir(jc_tract, home_tract):
    jc, ft = tract_cents.get(jc_tract, {}), tract_cents.get(home_tract, {})
    if not jc or not ft: return None
    angle = degrees(atan2(ft['lon'] - jc['lon'], ft['lat'] - jc['lat'])) % 360
    return ['N','NE','E','SE','S','SW','W','NW'][int((angle + 22.5) / 45) % 8]

def dist_mi(t1, t2):
    c1, c2 = tract_cents.get(t1, {}), tract_cents.get(t2, {})
    if not c1 or not c2: return None
    return ((c1['lat']-c2['lat'])**2 + (c1['lon']-c2['lon'])**2)**0.5 * 69

print('Proposed Route Corridors')
print('=' * 60)

for _, jc in top_jc.head(3).iterrows():
    jc_tract = jc['Job Center Tract']
    feeders = (stranded_df[stranded_df['work_tract'] == jc_tract]
              .groupby('home_tract').agg(
                  workers=('total_workers', 'sum'),
                  low_wage=('low_wage', 'sum')
              ).reset_index())
    feeders = feeders[feeders['workers'] >= 10]
    feeders['dist'] = feeders['home_tract'].apply(lambda t: dist_mi(t, jc_tract))
    feeders['dir'] = feeders['home_tract'].apply(lambda t: compass_dir(jc_tract, t))
    feeders = feeders.dropna().query('dist <= 20')

    print(f'\nJob Center: {jc_tract} ({int(jc["Stranded Workers"]):,} stranded workers)')

    for direction, group in feeders.groupby('dir'):
        top_stops = group.nlargest(8, 'workers').sort_values('dist')
        corridor_workers = top_stops['workers'].sum()
        if corridor_workers < 100: continue

        print(f'\n  {direction} Corridor: {corridor_workers:,} workers, {len(top_stops)} stops')
        for _, f in top_stops.iterrows():
            print(f'    ...{f["home_tract"][-4:]}  {int(f["workers"]):>5,} workers  {f["dist"]:>5.1f} mi')

## Summary

**Method:** For every worker in Dallas County (LODES), we check if DART's bus network (GTFS) can connect their home to their workplace — 0.5mi walk to/from stops, up to 2 transfers (transfers must share a stop within ~500ft).

**Key finding:** The majority of Dallas commuters have no viable transit path to their job. Workers are concentrated in clusters heading toward a small number of job centers.

**Route recommendations:** For each top job center, we group stranded workers by compass direction. Each direction = a proposed bus corridor with tracts ordered by distance (the stop sequence).

**Next steps:**
- Download GTFS for Fort Worth, Houston, San Antonio to repeat analysis
- Integrate into interactive Streamlit dashboard (app.py)
- Use SLD equity variables (zero-car HH, low-wage share) to prioritize corridors
- Compare with clustering analysis to validate neighborhood typologies